[Описание задачи](https://cups.online/ru/training/8/tasks/1367)

# Установка зависимостей

In [ ]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933264 sha256=2693dd2a0c70da3882549b878ef3ca79409647abb8aeb241fc40e212ab0d5813
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


# Импорты

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix

# Работа с данными

In [ ]:
df = pd.read_csv('train.csv')
df.head(5)

,user_id,course_id
0,39972,34
1,56815,51
2,63734,20
3,17896,81
4,36961,64


In [ ]:
df.shape

(10272, 2)

In [ ]:
print(df['user_id'].nunique(), df['course_id'].nunique())

4862 171


Почистим пользователей и курсы с количеством взаимодействий, меньше 2

In [ ]:
from copy import deepcopy, copy

def filter_column(df, min_freq):
    df = deepcopy(df)
    for col in df.columns:
        grouped = df.groupby(col).agg(
            freq=(col, 'count')
        ).reset_index()
        grouped_filtered = grouped[grouped['freq'] >= min_freq]
        df = df.loc[df[col].isin(grouped_filtered[col])]

    return df

def filter_dataframe(df, cols, min_freq):

    df_to_filter = df[cols].copy()

    while True:
        start_len = df_to_filter.shape[0]

        df_to_filter = filter_column(df_to_filter, min_freq)

        if df_to_filter.shape[0] == start_len:
            break

    for col in df_to_filter.columns:
        df = df.loc[df[col].isin(df_to_filter[col])]

    return df

In [ ]:
df_filtered = filter_dataframe(df, ['user_id', 'course_id'], 4)
df_filtered.shape

(3944, 2)

In [ ]:
df_filtered.head(5)

,user_id,course_id
3,17896,81
4,36961,64
8,68308,13
9,62349,134
10,11072,10


Узнаем, какие топ-3 курса являются самыми популярным, чтобы рекомендовать их новым холодным пользователям

In [ ]:
most_popular = df_filtered['course_id'].value_counts()
most_popular

,count
course_id,
7,190
1,171
15,169
51,128
43,123
...,...
166,5
221,4
196,4


In [ ]:
most_popular_courses_id = [7, 1, 15]

Кодирование пользователей и курсов

In [ ]:
user2idx = {u: i for i, u in enumerate(df_filtered['user_id'].unique())}
course2idx = {item: i for i, item in enumerate(df_filtered['course_id'].unique())}

idx2user = {i: u for u, i in user2idx.items()}
idx2course = {i: item for item, i in course2idx.items()}

In [ ]:
most_popular_courses_id = [course2idx[course_id] for course_id in most_popular_courses_id]
most_popular_courses_id

[34, 17, 40]

In [ ]:
df_filtered['user_idx'] = df_filtered['user_id'].map(user2idx)
df_filtered['course_idx'] = df_filtered['course_id'].map(course2idx)

Построение разреженной матрицы интеракций

In [ ]:
n_users = df_filtered['user_idx'].nunique()
n_courses = df_filtered['course_idx'].nunique()
n_interactions = df_filtered.shape[0]

interactions = csr_matrix(
    (
        np.ones(n_interactions),
        (df_filtered['user_idx'], df_filtered['course_idx'])
    ),
    shape=(n_users, n_courses)
)

In [ ]:
interactions.shape

(667, 128)

In [ ]:
from implicit.evaluation import leave_k_out_split

train, test = leave_k_out_split(
    interactions,
    K=1
)

In [ ]:
from implicit.nearest_neighbours import bm25_weight

train_bm25 = bm25_weight(train).tocsr()

# Модель и подбор гиперпараметров

Проинициализируем метрики для валидации модели и подбора гиперпараметров.

In [ ]:
from typing import List, Optional

def ap_metric(
    gt_items: List[int],
    predictions: List[int],
    topk: Optional[int]=None
):
    gt_items = np.asarray(gt_items)
    predictions = np.asarray(predictions)

    if len(gt_items) == 0:
        return 0.0

    if topk is not None:
        predictions = predictions[:topk]
        k = min(topk, len(gt_items))
    else:
        k = len(gt_items)

    relevance = np.isin(predictions, gt_items).astype(int)
    cumsum = np.cumsum(relevance)

    precision = cumsum / np.arange(1, len(relevance) + 1)

    ap = (1 / k) * np.sum(relevance * precision)

    return ap


def map_metric(
    gt_items_list: List[List[int]],
    predictions_list: List[List[int]],
    topk: Optional[int] = None
) -> float:

    if not gt_items_list or not predictions_list:
        return 0.0

    if len(gt_items_list) != len(predictions_list):
        raise ValueError("Количество пользователей в gt и predictions должно совпадать")

    ap_scores = []
    for gt, pred in zip(gt_items_list, predictions_list):
        ap = ap_metric(gt, pred, topk)
        ap_scores.append(ap)

    return np.mean(ap_scores) if ap_scores else 0.0

def hr_metric(
    gt_items: List[int],
    predictions: List[int],
    topk: Optional[int]=None
):

    if len(gt_items) == 0:
        return 0

    if topk is not None:
        predictions = predictions[:topk]

    relevance = np.isin(predictions, gt_items).astype(int)

    if np.sum(relevance) > 0:
        return 1
    return 0

Baseline MostPopular

In [ ]:
from collections import Counter

K = 3

item_popularity = np.asarray(train.sum(axis=0)).ravel()
popular_items = np.argsort(-item_popularity)

test_users = np.where(test.getnnz(axis=1) > 0)[0]

predictions_list = []
gt_items_list = []

for u in test_users:
    seen = set(train[u].indices)
    pred = [item for item in popular_items if item not in seen][:K]
    gt = test[u].indices.tolist()

    predictions_list.append(pred)
    gt_items_list.append(gt)

hr_scores = [hr_metric(gt, pred, topk=K) for gt, pred in zip(gt_items_list, predictions_list)]
ap_scores = [ap_metric(gt, pred, topk=K) for gt, pred in zip(gt_items_list, predictions_list)]

print(f"Popularity HR@{K} = {np.mean(hr_scores):.4f}")
print(f"Popularity MAP@{K} = {np.mean(ap_scores):.4f}")

Popularity HR@3 = 0.1700
Popularity MAP@3 = 0.1090


Подбор гиперпараметров

In [ ]:
from tqdm import tqdm
from itertools import product
from implicit.cpu.als import AlternatingLeastSquares
import pandas as pd
import numpy as np

factors_grid = [16, 32]
alpha_grid = [1, 5, 10]
reg_grid = [0.01, 0.05, 0.1, 0.5]
iter_grid = [10, 20, 30, 40]

K = 3
results = []

test_users = np.where(test.getnnz(axis=1) > 0)[0]

for f, a, it, r in tqdm(list(product(factors_grid, alpha_grid, iter_grid, reg_grid))):
    model = AlternatingLeastSquares(
        factors=f,
        regularization=r,
        alpha=a,
        iterations=it,
        random_state=42
    )

    model.fit(train_bm25)

    rec_items, _ = model.recommend(
        userid=test_users,
        user_items=train_bm25[test_users],
        N=3,
        filter_already_liked_items=True
    )

    if len(test_users) == 1:
        rec_items = np.expand_dims(rec_items, axis=0)

    predictions_list = rec_items.tolist()

    gt_items_list = [test[u].indices.tolist() for u in test_users]

    hr_scores = []
    ap_scores = []

    for gt, pred in zip(gt_items_list, predictions_list):
        hr_scores.append(hr_metric(gt, pred, topk=K))
        ap_scores.append(ap_metric(gt, pred, topk=K))

    hr_k = np.mean(hr_scores)
    map_k = np.mean(ap_scores)

    results.append({
        'factors': f,
        'alpha': a,
        'reg': r,
        'iterations': it,
        f'HR@{K}': hr_k,
        f'MAP@{K}': map_k
    })

    print(
        f"factors={f}, alpha={a}, iterations={it}, reg={r} | "
        f"HR@{K}={hr_k:.4f}, MAP@{K}={map_k:.4f}"
    )

results_df = pd.DataFrame(results).sort_values(
    by=[f'MAP@{K}', f'HR@{K}'],
    ascending=False
).reset_index(drop=True)

print("\nЛучшие гиперпараметры:")
print(results_df.head(10))

  0%|          | 0/96 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  1%|          | 1/96 [00:00<00:40,  2.35it/s]

factors=16, alpha=1, iterations=10, reg=0.01 | HR@3=0.2534, MAP@3=0.1870


  0%|          | 0/10 [00:00<?, ?it/s]

...

 99%|█████████▉| 95/96 [01:18<00:01,  1.28s/it]

factors=32, alpha=10, iterations=40, reg=0.1 | HR@3=0.1818, MAP@3=0.1348


  0%|          | 0/40 [00:00<?, ?it/s]

100%|██████████| 96/96 [01:20<00:00,  1.19it/s]

factors=32, alpha=10, iterations=40, reg=0.5 | HR@3=0.1827, MAP@3=0.1362

Лучшие гиперпараметры:
   factors  alpha   reg  iterations      HR@3     MAP@3
0       16      1  0.05          30  0.255051  0.190937
1       16      1  0.10          30  0.254209  0.190376
2       16      1  0.01          30  0.259259  0.190095
3       16      1  0.50          30  0.257576  0.189254
4       16      1  0.50          10  0.250842  0.188692
5       16      1  0.10          40  0.255051  0.188552
6       16      1  0.01          20  0.250842  0.188412
7       16      1  0.01          40  0.255892  0.188131
8       16      1  0.10          10  0.254209  0.188131
9       16      1  0.50          20  0.251684  0.187991


In [ ]:
print("\nЛучшие гиперпараметры:")
print(results_df.head(1))


Лучшие гиперпараметры:
   factors  alpha   reg  iterations      HR@3     MAP@3
0       16      1  0.05          30  0.255051  0.190937


Итого HR@3 удалось повысить на 50%, а MAP@3 на 91%